In [ ]:
import pandas as pd

mini_vanilla_df, full_vanilla_responses, vanilla_codes, vanilla_rewards = pd.read_pickle("vanilla_generate_responses_outputs.pkl")
mini_spartan_df, full_spartan_responses, spartan_codes, spartan_rewards = pd.read_pickle("spartan_generate_responses_outputs.pkl")

In [ ]:
import ast


def pass_at_k(n: int, c: int, k: int) -> float:
    if n - c < k:
        return 1.0

    # Total ways to choose k samples from n
    total_combinations = math.comb(n, k)
    
    # Ways to choose k samples ONLY from the incorrect (n - c) samples
    incorrect_combinations = math.comb(n - c, k)
    
    # The probability that ALL k chosen samples are incorrect
    prob_all_incorrect = incorrect_combinations / total_combinations
    
    # Pass@k = 1 - (Probability that all k are incorrect)
    pass_at_k_value = 1.0 - prob_all_incorrect
    
    return pass_at_k_value

def compute_pass_at_k(rewards, k=1, group_size=8):
    c = int(sum(rewards))
    n = group_size
    return pass_at_k(n, c, k)

def calculate_unique_variable_count(code_string):
    """
    Counts the number of unique variables assigned a value in the code block.
    This measures the Spartan principle of minimizing the number of variables.
    """
    try:
        tree = ast.parse(code_string)
    except SyntaxError:
        return -1 # Indicate syntax error

    assigned_vars = set()

    for node in ast.walk(tree):
        # Look for assignments (e.g., x = 5)
        if isinstance(node, ast.Assign):
            for target in node.targets:
                if isinstance(target, ast.Name):
                    assigned_vars.add(target.id)
        # Look for function arguments (e.g., def func(x, y):)
        elif isinstance(node, ast.FunctionDef):
            for arg in node.args.args:
                assigned_vars.add(arg.arg)
        # Look for 'for' loop targets (e.g., for i in range(5):)
        elif isinstance(node, ast.For):
             if isinstance(node.target, ast.Name):
                assigned_vars.add(node.target.id)

    return len(assigned_vars)

def calculate_cyclomatic_complexity(code_string):
    """
    Calculates the Cyclomatic Complexity (M).
    This directly measures the Spartan principle of minimizing control structures.
    """
    try:
        tree = ast.parse(code_string)
    except SyntaxError:
        return -1 # Indicate syntax error

    # Start with 1 (the single entry/exit path)
    complexity = 1

    # Decision points increase complexity by 1
    # Note: ast.For, ast.While, ast.If, ast.ExceptHandler are main decision points
    # ast.BoolOp (and, or) also increases complexity
    for node in ast.walk(tree):
        if isinstance(node, (ast.If, ast.While, ast.For, ast.ExceptHandler, ast.With)):
            complexity += 1
        elif isinstance(node, ast.BoolOp):
            # 'and'/'or' operators contribute (k-1) where k is the number of values.
            # E.g., a and b and c has 2 decision points.
            complexity += len(node.values) - 1
        
        # We don't count function definitions as decision points here, 
        # as we are measuring complexity *within* the code block.

    return complexity

def calculate_halstead_volume(code_string):
    """
    Calculates the Halstead Volume, a measure of program size and vocabulary.
    Minimizing this value aligns with Spartan 'Token Count' minimization.
    """
    try:
        tree = ast.parse(code_string)
    except SyntaxError:
        return -1 # Indicate syntax error

    operators = set()
    operands = set()

    for node in ast.walk(tree):
        # Identify Operators (e.g., BinOp, Compare, Assign, FunctionDef)
        if isinstance(node, (ast.BinOp, ast.UnaryOp, ast.Compare, ast.BoolOp)):
            if hasattr(node, 'op'):
                operators.add(type(node.op).__name__)
        elif isinstance(node, (ast.FunctionDef, ast.ClassDef)):
            operators.add(type(node).__name__) # Treat definition as an operator

        # Identify Operands (e.g., constants, names, attribute values)
        elif isinstance(node, (ast.Constant, ast.Name)):
            if isinstance(node, ast.Name) and isinstance(node.ctx, ast.Load):
                # Only count Names when they are loaded/used, not just stored/assigned to.
                operands.add(node.id)
            elif isinstance(node, ast.Constant):
                operands.add(repr(node.value)) # Use repr for unique identification

    N1 = len(operators) # Total operators (approximate)
    N2 = len(operands)  # Total operands (approximate)
    n1 = len(operators) # Unique operators
    n2 = len(operands)  # Unique operands

    if n1 + n2 == 0:
        return 0.0

    # Halstead Volume Calculation
    V = (N1 + N2) * (n1 + n2).bit_length() / 8 # Approximation for log2(n1+n2)
    return V

def calculate_source_lines_of_code(code_string: str) -> int:
    """
    Calculates Source Lines of Code (SLOC).
    This measures the Spartan principle of minimizing vertical complexity (code length).

    It counts lines that are not blank and are not just comments.
    """
    lines = code_string.split('\n')
    sloc_count = 0
    
    # Simple approach to filter out blank and pure comment lines
    for line in lines:
        stripped_line = line.strip()
        
        # Check for blank line or pure comment line
        if not stripped_line or stripped_line.startswith('#'):
            continue
        
        # If the line contains anything else (code, comment after code), count it.
        sloc_count += 1
        
    return sloc_count

def calculate_cognitive_complexity(code_string: str) -> int:
    """
    Calculates the Cognitive Complexity of the code string based on SonarSource rules.
    This measures the Spartan principle of reducing cognitive load and nesting.
    """
    try:
        tree = ast.parse(code_string)
    except SyntaxError:
        return -1 # Indicate syntax error
    
    complexity = 0
    
    # Structure to hold complexity and nesting level for each block
    class ComplexityVisitor(ast.NodeVisitor):
        def __init__(self):
            self.complexity = 0
            self.nesting_level = 0
            # Nodes that increase complexity and nesting
            self.nesting_nodes = (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef, ast.For, ast.AsyncFor, ast.While, ast.If, ast.With, ast.AsyncWith)
            # Nodes that only increase complexity (not nesting)
            self.increment_nodes = (ast.Try, ast.ExceptHandler, ast.Yield, ast.YieldFrom, ast.Break, ast.Continue)

        def visit(self, node):
            # 1. Nesting Increment Rule (must happen before calling generic_visit)
            if isinstance(node, self.nesting_nodes) and not isinstance(node, ast.If):
                # Function/Class/Loops/With start a new logical block and increase nesting
                self.nesting_level += 1
                
            # 2. Increment Rule: +1 for flow-breaking structures, plus nesting cost
            if isinstance(node, self.nesting_nodes) or isinstance(node, self.increment_nodes):
                increment_cost = 1
                
                # Special handling for 'elif' in ast.If: only the top 'if' increases nesting/complexity
                # The 'If' node structure in AST handles if/elif/else under one node.
                if isinstance(node, ast.If):
                    # Only count the initial 'if' for complexity, not the 'elif's implicitly handled inside
                    if self.nesting_level == 0:
                        increment_cost = 1 
                    else:
                        increment_cost = 1 + self.nesting_level # Add nesting cost

                # 'try', 'except', 'yield', 'break', 'continue' are flat increments of +1
                elif isinstance(node, self.increment_nodes):
                    increment_cost = 1
                    
                # All other nesting nodes (Function, For, While, With) get nesting cost
                else:
                    increment_cost = 1 + self.nesting_level
                    
                self.complexity += increment_cost

            # Recurse into children
            self.generic_visit(node)
            
            # 3. Nesting Decrement Rule (must happen after generic_visit)
            if isinstance(node, self.nesting_nodes) and not isinstance(node, ast.If):
                self.nesting_level -= 1

    # Apply the visitor starting from the root of the AST
    visitor = ComplexityVisitor()
    visitor.visit(tree)
    return visitor.complexity


In [ ]:
from transformers import AutoTokenizer
import numpy as np
import math

pd.options.mode.chained_assignment = None

group_size = 8

# Work on copies to avoid changing original dataframes
mini_vanilla_df = mini_vanilla_df.copy()
mini_spartan_df = mini_spartan_df.copy()

# Repeat each row n times
flattened_mini_vanilla_df = mini_vanilla_df.loc[mini_vanilla_df.index.repeat(group_size)].reset_index(drop=True)
flattened_mini_spartan_df = mini_spartan_df.loc[mini_spartan_df.index.repeat(group_size)].reset_index(drop=True)

# Add a repeat_index where the first n rows are 0, the next n are 1, etc.
flattened_mini_vanilla_df['repeat_index'] = [i // group_size for i in range(len(flattened_mini_vanilla_df))]
flattened_mini_spartan_df['repeat_index'] = [i // group_size for i in range(len(flattened_mini_spartan_df))]

flattened_mini_vanilla_df["code"] = vanilla_codes
flattened_mini_spartan_df["code"] = spartan_codes
flattened_mini_vanilla_df["reward"] = vanilla_rewards
flattened_mini_spartan_df["reward"] = spartan_rewards

grouped_mini_vanilla_df = flattened_mini_vanilla_df.groupby("repeat_index").agg({
    "code": list,
    "reward": list
}).reset_index()
grouped_mini_spartan_df = flattened_mini_spartan_df.groupby("repeat_index").agg({
    "code": list,
    "reward": list
}).reset_index()


tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-4B", trust_remote_code=True)

grouped_mini_vanilla_df["unique_variable_count"] = grouped_mini_vanilla_df["code"].apply(lambda x: [calculate_unique_variable_count(code) for code in x if code is not None])
grouped_mini_spartan_df["unique_variable_count"] = grouped_mini_spartan_df["code"].apply(lambda x: [calculate_unique_variable_count(code) for code in x if code is not None])

grouped_mini_vanilla_df["cyclomatic_complexity"] = grouped_mini_vanilla_df["code"].apply(lambda x: [calculate_cyclomatic_complexity(code) for code in x if code is not None])
grouped_mini_spartan_df["cyclomatic_complexity"] = grouped_mini_spartan_df["code"].apply(lambda x: [calculate_cyclomatic_complexity(code) for code in x if code is not None])

grouped_mini_vanilla_df["halstead_volume"] = grouped_mini_vanilla_df["code"].apply(lambda x: [calculate_halstead_volume(code) for code in x if code is not None])
grouped_mini_spartan_df["halstead_volume"] = grouped_mini_spartan_df["code"].apply(lambda x: [calculate_halstead_volume(code) for code in x if code is not None])

grouped_mini_vanilla_df["sloc"] = grouped_mini_vanilla_df["code"].apply(lambda x: [calculate_source_lines_of_code(code) for code in x if code is not None])
grouped_mini_spartan_df["sloc"] = grouped_mini_spartan_df["code"].apply(lambda x: [calculate_source_lines_of_code(code) for code in x if code is not None])

grouped_mini_vanilla_df["cognitive_complexity"] = grouped_mini_vanilla_df["code"].apply(lambda x: [calculate_cognitive_complexity(code) for code in x if code is not None])
grouped_mini_spartan_df["cognitive_complexity"] = grouped_mini_spartan_df["code"].apply(lambda x: [calculate_cognitive_complexity(code) for code in x if code is not None])

filtered_grouped_mini_vanilla_df = grouped_mini_vanilla_df[grouped_mini_vanilla_df["code"].apply(lambda codes: any(code is not None for code in codes))]
filtered_grouped_mini_spartan_df = grouped_mini_spartan_df[grouped_mini_spartan_df["code"].apply(lambda codes: any(code is not None for code in codes))]

filtered_grouped_mini_vanilla_df["pass@1"] = filtered_grouped_mini_vanilla_df["reward"].apply(lambda rewards: compute_pass_at_k(rewards, k=1, group_size=group_size))
filtered_grouped_mini_spartan_df["pass@1"] = filtered_grouped_mini_spartan_df["reward"].apply(lambda rewards: compute_pass_at_k(rewards, k=1, group_size=group_size))


joined_df = filtered_grouped_mini_vanilla_df.merge(
    filtered_grouped_mini_spartan_df,
    on="repeat_index",
    suffixes=("_vanilla", "_spartan"),
    how="inner"
)
joined_df.head()

In [ ]:
import plotly.graph_objs as go
import plotly.subplots as sp
import plotly.express as px

# Make generic list of metrics for analysis with relevant plotting/label info
# Remove avg_code_length/code_length from plotting/metrics
metrics = [
    {
        "name": "pass@1",
        "vanilla_col": "pass@1_vanilla",
        "spartan_col": "pass@1_spartan",
        "title": "Pass@1 Scores (per question)",
        "yaxis": "Pass@1",
        "vanilla_color": "orange",
        "spartan_color": "red"
    },
    {
        "name": "unique_variable_count",
        "vanilla_col": "unique_variable_count_vanilla",
        "spartan_col": "unique_variable_count_spartan",
        "title": "Unique Variable Count (per question)",
        "yaxis": "Avg Unique Variable Count",
        "vanilla_color": "purple",
        "spartan_color": "magenta"
    },
    {
        "name": "cyclomatic_complexity",
        "vanilla_col": "cyclomatic_complexity_vanilla",
        "spartan_col": "cyclomatic_complexity_spartan",
        "title": "Cyclomatic Complexity (per question)",
        "yaxis": "Avg Cyclomatic Complexity",
        "vanilla_color": "brown",
        "spartan_color": "olive"
    },
    {
        "name": "halstead_volume",
        "vanilla_col": "halstead_volume_vanilla",
        "spartan_col": "halstead_volume_spartan",
        "title": "Halstead Volume (per question)",
        "yaxis": "Avg Halstead Volume",
        "vanilla_color": "darkblue",
        "spartan_color": "darkgreen"
    },
    {
        "name": "sloc",
        "vanilla_col": "sloc_vanilla",
        "spartan_col": "sloc_spartan",
        "title": "SLOC (per question)",
        "yaxis": "Avg SLOC",
        "vanilla_color": "teal",
        "spartan_color": "limegreen"
    },
    {
        "name": "cognitive_complexity",
        "vanilla_col": "cognitive_complexity_vanilla",
        "spartan_col": "cognitive_complexity_spartan",
        "title": "Cognitive Complexity (per question)",
        "yaxis": "Avg Cognitive Complexity",
        "vanilla_color": "navy",
        "spartan_color": "gold"
    }
]

# Compute per-question means for each model/metric
for met in metrics:
    for model in ["vanilla", "spartan"]:
        col = met[f"{model}_col"]
        # Special handling for pass@1, which is a float and not a list
        if met["name"] == "pass@1":
            avg_col = col
            # No need to compute an average, value is already a float
            if avg_col not in joined_df.columns:
                print(f"{avg_col} not in joined_df, skipping.")
            # Always set the avg_col keys for plotting code below
            met[f"{model}_avg_col"] = avg_col
        else:
            avg_col = f"avg_{met['name']}_{model}" if not col.startswith("avg_") else col
            if avg_col not in joined_df.columns or not col.startswith("avg_"):
                print(col)
                joined_df[avg_col] = joined_df[col].apply(lambda x: np.mean(x) if isinstance(x, (list, np.ndarray)) and len(x) else np.nan)
            met[f"{model}_avg_col"] = avg_col

# Stacked plots: one row per metric
num_metrics = len(metrics)
fig = sp.make_subplots(
    rows=num_metrics, cols=1,
    subplot_titles=tuple(met["title"] for met in metrics),
    shared_xaxes=True,
    vertical_spacing=0.08
)

for idx, met in enumerate(metrics, 1):
    # Add vanilla trace
    fig.add_trace(
        go.Scatter(
            x=joined_df["repeat_index"],
            y=joined_df[met['vanilla_avg_col']],
            mode='markers+lines',
            name=f"Vanilla {met['title'].split('(')[0].strip()}",
            marker=dict(color=met["vanilla_color"])
        ),
        row=idx, col=1
    )
    # Add spartan trace
    fig.add_trace(
        go.Scatter(
            x=joined_df["repeat_index"],
            y=joined_df[met['spartan_avg_col']],
            mode='markers+lines',
            name=f"Spartan {met['title'].split('(')[0].strip()}",
            marker=dict(color=met["spartan_color"])
        ),
        row=idx, col=1
    )
    fig.update_yaxes(title_text=met["yaxis"], row=idx, col=1)

fig.update_xaxes(title_text="question_idx", row=num_metrics, col=1)
fig.update_layout(height=340*num_metrics, width=900, showlegend=True)
fig.show()

# --- Aggregated bar plots, generic loop for metrics
for met in metrics:
    # Only do bar plots where values are floats (skip metrics with unusual types if any)
    vanilla_vals = joined_df[met["vanilla_avg_col"]]
    spartan_vals = joined_df[met["spartan_avg_col"]]
    try:
        avg_vanilla = vanilla_vals.mean()
        avg_spartan = spartan_vals.mean()
        bar_fig = px.bar(
            x=["Vanilla", "Spartan"],
            y=[avg_vanilla, avg_spartan],
            labels={'x': 'Model', 'y': f'Avg {met["yaxis"]}'},
            title=f'Aggregated Mean {met["title"].split("(")[0].strip()} Comparison',
            color=["Vanilla", "Spartan"],
            color_discrete_map={"Vanilla": met["vanilla_color"], "Spartan": met["spartan_color"]}
        )
        bar_fig.show()
    except Exception as e:
        print(f"Could not plot bar for {met['name']}: {e}")

